In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [26]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [28]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [29]:
for i, image in enumerate(decoded_preds, 1):
    print(i)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
